In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True


In [2]:
sys.path.append('/content/drive/MyDrive')

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 0  —  Wajib dirun PERTAMA
#   1. Hapus cache lama outputs_ablation_features/  (agar hasil fresh)
#   2. Generate common.py ke /content/  (dari common.ipynb yang sudah ada)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import shutil, os, json, sys

# 1. Hapus cache lama
OUTDIR = 'outputs_ablation_features'
if os.path.exists(OUTDIR):
    shutil.rmtree(OUTDIR)
    print(f'✅ Cache lama dihapus: {OUTDIR}/')
else:
    print(f'ℹ️  Belum ada cache — fresh run')

# 2. Convert common.ipynb → common.py agar bisa diimport
#    Cari common.ipynb di /content/drive/MyDrive/ atau /content/
_possible = [
    '/content/common.ipynb',
    '/content/drive/MyDrive/common.ipynb',
    '/content/drive/MyDrive/Detector/common.ipynb',
]
_nb_path = next((p for p in _possible if os.path.exists(p)), None)

if _nb_path:
    print(f'✅ Ditemukan: {_nb_path}')
    with open(_nb_path) as _f:
        _nb = json.load(_f)
    _parts = []
    for _c in _nb['cells']:
        if _c['cell_type'] == 'code':
            _src = ''.join(_c['source']).rstrip()
            if _src:
                _parts.append(_src)
    _common_py = '\n\n'.join(_parts)
    with open('/content/common.py', 'w') as _f:
        _f.write(_common_py)
    print('✅ /content/common.py berhasil di-generate dari common.ipynb')
else:
    # Fallback: embed langsung dari sini
    print('⚠️  common.ipynb tidak ditemukan di Drive — pakai embedded fallback')
    _common_py = '"""\ncommon.py — Shared pipeline functions for the revision experiments.\n\nThese functions are copied VERBATIM from the author\'s notebooks\n(AMSTE_25ms.ipynb / AMSTE_15ms.ipynb) so that every revision experiment\nuses exactly the same pseudo-episode construction, episode split, windowing,\ntabular-feature extraction, and metric computation as the original paper.\nDo NOT modify these unless you also re-run the main results.\n\nTwo datasets are used by the original code:\n    25 m/s  ->  Dataset_1.csv\n    15 m/s  ->  Dataset_2.csv\nSet the paths in CONFIG below (or pass them to load_dataset()).\n"""\nimport os, json, glob, random\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import (accuracy_score, precision_recall_fscore_support,\n                             confusion_matrix)\n\n# ─────────────────────────────────────────────────────────────────────────\n# CONFIG  (verbatim values from the notebooks)\n# ─────────────────────────────────────────────────────────────────────────\nSEEDS          = [42, 123, 456, 789, 2024, 7, 13, 21, 33, 55, 77, 99, 111, 222, 333, 444, 555, 777, 888, 999]\nFEATURES       = [\'SNR\', \'RSSI\', \'PDR\', \'Speed\', \'Relative_Speed\']\nWINDOW_SIZES   = [1, 5, 15]\nEPISODE_LENGTH = 60          # seconds per pseudo-episode\nWINDOW_SEC     = 15          # window used by RF-Stat / XGBoost-only / LSTM\nK_NEIGHBORS    = 5\nSPEED_RANGE    = {15: (13, 17), 25: (23, 27)}\nSCENARIO_TO_LABEL = {1: 0, 2: 1, 3: 2}   # Interference / Reactive / Constant\n\n# EDIT THESE to point at your local copies of the dataset:\nDATASET_PATHS = {\n    25: os.environ.get(\'DATASET_25\', \'Dataset_1.csv\'),\n    15: os.environ.get(\'DATASET_15\', \'Dataset_2.csv\'),\n}\n\n# ─────────────────────────────────────────────────────────────────────────\n# Dataset loading  (mirrors notebook Cell 2)\n# ─────────────────────────────────────────────────────────────────────────\ndef load_dataset(target_speed, path=None):\n    """Load CSV, map labels, build pseudo_run_id, filter to one speed band."""\n    if path is None:\n        path = DATASET_PATHS[target_speed]\n    if not os.path.exists(path):\n        raise FileNotFoundError(\n            f"Dataset for {target_speed} m/s not found at \'{path}\'. "\n            f"Set DATASET_PATHS[{target_speed}] in common.py or the "\n            f"DATASET_{target_speed} environment variable.")\n    df = pd.read_csv(path)\n    df[\'label\'] = df[\'Scenario\'].map(SCENARIO_TO_LABEL)\n    df = df.sort_values([\'Speed\', \'Scenario\', \'Time\']).reset_index(drop=True)\n    df[\'pseudo_run_id\'] = (\n        df[\'Speed\'].round(2).astype(str) + \'_\' +\n        df[\'Scenario\'].astype(str) + \'_\' +\n        (df[\'Time\'] // EPISODE_LENGTH).astype(int).astype(str)\n    )\n    lo, hi = SPEED_RANGE[target_speed]\n    df_speed = df[(df[\'Speed\'] >= lo) & (df[\'Speed\'] <= hi)].copy()\n    return df_speed\n\ndef set_all_seeds(seed):\n    np.random.seed(seed)\n    random.seed(seed)\n    os.environ[\'PYTHONHASHSEED\'] = str(seed)\n    try:\n        import tensorflow as tf\n        tf.random.set_seed(seed)\n    except Exception:\n        pass\n\n# ─────────────────────────────────────────────────────────────────────────\n# Episode split  (VERBATIM from notebook)\n# ─────────────────────────────────────────────────────────────────────────\ndef split_by_episode(df, train_ratio=0.6, val_ratio=0.2, seed=42):\n    """Stratified episode-based split (per-class shuffle)."""\n    rng = np.random.RandomState(seed)\n    ep_to_label = (df.groupby(\'pseudo_run_id\')[\'label\']\n                     .agg(lambda x: x.iloc[0]).to_dict())\n    eps_by_class = {}\n    for ep, lbl in ep_to_label.items():\n        eps_by_class.setdefault(lbl, []).append(ep)\n\n    train_eps, val_eps, test_eps = [], [], []\n    for lbl, eps in eps_by_class.items():\n        eps = np.array(eps)\n        rng.shuffle(eps)\n        n = len(eps)\n        n_train = int(round(train_ratio * n))\n        n_val = int(round(val_ratio * n))\n        if n >= 3:\n            n_train = max(1, min(n - 2, n_train))\n            n_val = max(1, min(n - n_train - 1, n_val))\n        train_eps.extend(eps[:n_train].tolist())\n        val_eps.extend(eps[n_train:n_train + n_val].tolist())\n        test_eps.extend(eps[n_train + n_val:].tolist())\n\n    train_df = df[df[\'pseudo_run_id\'].isin(train_eps)].copy()\n    val_df   = df[df[\'pseudo_run_id\'].isin(val_eps)].copy()\n    test_df  = df[df[\'pseudo_run_id\'].isin(test_eps)].copy()\n    assert not (set(train_eps) & set(test_eps)), \'LEAKAGE!\'\n    assert not (set(train_eps) & set(val_eps)),  \'LEAKAGE!\'\n    assert not (set(val_eps)   & set(test_eps)), \'LEAKAGE!\'\n    return train_df, val_df, test_df\n\n# ─────────────────────────────────────────────────────────────────────────\n# Windowing  (VERBATIM — non-overlapping, step = window length)\n# ─────────────────────────────────────────────────────────────────────────\ndef create_windows(df, window_sec, step_sec=None):\n    if step_sec is None:\n        step_sec = window_sec\n    X_list, y_list, ep_list = [], [], []\n    for run_id, group in df.groupby(\'pseudo_run_id\'):\n        group = group.sort_values(\'Time\').reset_index(drop=True)\n        times = group[\'Time\'].values\n        X_raw = group[FEATURES].values\n        y_raw = group[\'label\'].values\n        t0, t_end = times[0], times[-1]\n        while t0 + window_sec <= t_end:\n            t1 = t0 + window_sec\n            mask = (times >= t0) & (times < t1)\n            if mask.sum() > 0:\n                X_list.append(X_raw[mask])\n                y_list.append(np.bincount(y_raw[mask]).argmax())\n                ep_list.append(run_id)\n            t0 += step_sec\n    return X_list, np.array(y_list), ep_list\n\ndef pad_sequences(X_list, max_len=None):\n    if max_len is None:\n        max_len = max(len(x) for x in X_list)\n    n_feat = X_list[0].shape[1]\n    out = np.zeros((len(X_list), max_len, n_feat), dtype=np.float32)\n    for i, x in enumerate(X_list):\n        sl = min(len(x), max_len)\n        out[i, :sl, :] = x[:sl]\n    return out, max_len\n\ndef pad_and_normalize(X_train_list, X_val_list, X_test_list):\n    max_len = max(len(x) for x in X_train_list)\n    n_features = X_train_list[0].shape[1]\n    def pad(X_list):\n        out = np.zeros((len(X_list), max_len, n_features), dtype=np.float32)\n        for i, x in enumerate(X_list):\n            sl = min(len(x), max_len)\n            out[i, :sl, :] = x[:sl]\n        return out\n    Xtr, Xv, Xte = pad(X_train_list), pad(X_val_list), pad(X_test_list)\n    sc = StandardScaler()\n    Xtr = sc.fit_transform(Xtr.reshape(-1, n_features)).reshape(Xtr.shape)\n    Xv  = sc.transform(Xv.reshape(-1, n_features)).reshape(Xv.shape)\n    Xte = sc.transform(Xte.reshape(-1, n_features)).reshape(Xte.shape)\n    return Xtr, Xv, Xte, max_len\n\n# ─────────────────────────────────────────────────────────────────────────\n# Tabular features  (VERBATIM — 7 stats x 5 features = 35 dims)\n# ─────────────────────────────────────────────────────────────────────────\ndef extract_tabular_features(X_padded):\n    out = []\n    for x in X_padded:\n        m = ~(x == 0).all(axis=1)\n        xv = x[m] if m.sum() > 0 else x\n        f = []\n        for c in range(xv.shape[1]):\n            v = xv[:, c]\n            f.extend([np.mean(v), np.std(v), np.min(v), np.max(v),\n                      np.median(v), np.percentile(v, 25), np.percentile(v, 75)])\n        out.append(f)\n    return np.array(out)\n\n# 7 summary statistics per feature, in the order produced above:\nSTATS_PER_FEATURE = 7\ndef feature_block_indices(feature_name):\n    """Return the 7 column indices in the 35-dim vector for one raw feature."""\n    c = FEATURES.index(feature_name)\n    return list(range(c * STATS_PER_FEATURE, (c + 1) * STATS_PER_FEATURE))\n\ndef select_tabular_columns(X35, drop_features=()):\n    """Return a copy of the 35-dim tabular matrix with the columns belonging\n    to `drop_features` removed (used for the no-relative-speed ablations)."""\n    drop = set()\n    for fn in drop_features:\n        drop.update(feature_block_indices(fn))\n    keep = [i for i in range(X35.shape[1]) if i not in drop]\n    return X35[:, keep], keep\n\n# ─────────────────────────────────────────────────────────────────────────\n# Metrics  (VERBATIM)\n# ─────────────────────────────────────────────────────────────────────────\ndef compute_metrics(y_true, y_pred, num_classes=3):\n    acc = float(accuracy_score(y_true, y_pred))\n    p, r, f1, _ = precision_recall_fscore_support(\n        y_true, y_pred, average=\'macro\', zero_division=0)\n    labels = list(range(num_classes))\n    p_pc, r_pc, f1_pc, _ = precision_recall_fscore_support(\n        y_true, y_pred, labels=labels, average=None, zero_division=0)\n    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()\n    return dict(accuracy=acc, precision_macro=float(p),\n                recall_macro=float(r), f1_macro=float(f1),\n                precision_per_class=p_pc.tolist(),\n                recall_per_class=r_pc.tolist(),\n                f1_per_class=f1_pc.tolist(),\n                confusion_matrix=cm)\n\ndef agg(values, pct=True):\n    """mean ± sample-std (ddof=1) helper, matching the notebooks."""\n    a = np.asarray(values, dtype=float)\n    m, s = a.mean(), (a.std(ddof=1) if len(a) > 1 else 0.0)\n    if pct:\n        m, s = m * 100, s * 100\n    return m, s\n\nCLASS_NAMES = [\'Interference\', \'Reactive\', \'Constant\']'
    with open('/content/common.py', 'w') as _f:
        _f.write(_common_py)
    print('✅ /content/common.py berhasil ditulis (embedded fallback)')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')
    print('✅ /content ditambahkan ke sys.path')
print('\n→ Lanjut ke cell berikutnya.')


ℹ️  Belum ada cache — fresh run
⚠️  common.ipynb tidak ditemukan di Drive — pakai embedded fallback
✅ /content/common.py berhasil ditulis (embedded fallback)

→ Lanjut ke cell berikutnya.


In [4]:
"""
t2_feature_ablation.py  —  T2 (review items C2)

THE most informative experiment in the revision. Re-runs the two handcrafted-
feature configurations (RF-Stat and XGBoost-only) under three feature sets,
across all 20 seeds, at both regimes:

    (a) full            : all 5 features (35-dim)         -- reproduces the paper
    (b) no_relspeed     : drop Relative_Speed             -- (28-dim)
    (c) no_speed_feats  : drop Speed AND Relative_Speed   -- (21-dim)

Interpretation guide:
  * If accuracy at 25 m/s stays high without relative speed, the "dataset-level
    separability" reading is strengthened.
  * If accuracy collapses when relative speed is removed, separability is
    carried almost entirely by one feature -- which makes residual coupling /
    leakage a more plausible explanation and means the 100% claim must be
    softened harder than the review suggests.

Uses the SAME windowing, split, and 35-dim tabular features as the paper
(via common.py). Classifier hyper-parameters mirror the notebook:
    XGBoost-only : XGBClassifier(n_estimators=200, max_depth=5, lr=0.05, ...)
    RF-Stat      : RandomForestClassifier(n_estimators=200, max_depth=15, ...)
      NOTE: the paper text (Sec 3.5) says RF = 300 trees / depth 4, but the
      released code uses 200 / 15. This script follows the CODE. Reconcile the
      manuscript text with whichever you actually report.

NEEDS THE DATASET.
"""
import os, json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from common import (load_dataset, split_by_episode, create_windows,
                    pad_and_normalize, extract_tabular_features,
                    select_tabular_columns, compute_metrics, agg,
                    set_all_seeds, SEEDS, WINDOW_SEC)

FEATURE_SETS = {
    'full':           (),
    'no_relspeed':    ('Relative_Speed',),
    'no_speed_feats': ('Speed', 'Relative_Speed'),
}


def _tabular_for_split(train_df, val_df, test_df):
    Xtr_l, ytr, _ = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte, _ = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        return None
    # pad/normalize on the full 5-feature tensor (matches pipeline), then
    # extract 35-dim tabular features; column dropping happens afterwards.
    Xtr_s, Xv_s, Xte_s, _ = pad_and_normalize(Xtr_l, Xv_l, Xte_l)
    return (extract_tabular_features(Xtr_s), ytr,
            extract_tabular_features(Xv_s),  yv,
            extract_tabular_features(Xte_s), yte)


def run_one(df_speed, seed, drop_features):
    set_all_seeds(seed)
    tr, va, te = split_by_episode(df_speed, seed=seed)
    packed = _tabular_for_split(tr, va, te)
    if packed is None:
        return None
    Xtr35, ytr, Xv35, yv, Xte35, yte = packed
    Xtr, kept = select_tabular_columns(Xtr35, drop_features)
    Xv,  _    = select_tabular_columns(Xv35,  drop_features)
    Xte, _    = select_tabular_columns(Xte35, drop_features)

    # NOTE: NO StandardScaler — notebook does not apply one for tree models.
    # Tree-based models are scale-invariant; the notebook feeds raw tabular
    # features from extract_tabular_features() directly to XGBoost and RF.

    # --- RF-Stat (mirrors released code) ---
    rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                                min_samples_split=5, min_samples_leaf=2,
                                n_jobs=-1, random_state=seed,
                                class_weight='balanced')
    rf.fit(Xtr, ytr)
    rf_acc = compute_metrics(yte, rf.predict(Xte))['accuracy']

    # --- XGBoost-only (mirrors released code) ---
    xgb = XGBClassifier(objective='multi:softprob', num_class=3,
                        n_estimators=200, max_depth=5, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        random_state=seed, eval_metric='mlogloss',
                        early_stopping_rounds=20)
    xgb.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
    xgb_acc = compute_metrics(yte, xgb.predict(Xte))['accuracy']

    return {'rf_stat': rf_acc, 'xgb_only': xgb_acc, 'n_features': Xtr.shape[1]}


def main(outdir='outputs_ablation_features'):
    os.makedirs(outdir, exist_ok=True)
    results = {}
    for spd in (25, 15):
        df = load_dataset(spd)
        for set_name, drop in FEATURE_SETS.items():
            rf_accs, xgb_accs, nfeat = [], [], None
            for seed in SEEDS:
                r = run_one(df, seed, drop)
                if r:
                    rf_accs.append(r['rf_stat']); xgb_accs.append(r['xgb_only'])
                    nfeat = r['n_features']
            results[f'{spd}ms_{set_name}'] = {
                'speed': spd, 'feature_set': set_name, 'n_features': nfeat,
                'rf_stat_per_seed': rf_accs, 'xgb_only_per_seed': xgb_accs,
                'rf_stat_mean_std': list(map(lambda v: round(v, 2), agg(rf_accs))),
                'xgb_only_mean_std': list(map(lambda v: round(v, 2), agg(xgb_accs))),
            }

    print('\n' + '=' * 84)
    print('  T2 — Handcrafted-feature ablation (accuracy %, mean ± std across 20 seeds)')
    print('=' * 84)
    print(f'{"Regime":>8s}{"Feature set":>18s}{"#feat":>7s}{"RF-Stat":>16s}{"XGBoost-only":>18s}')
    print('-' * 84)
    for k, v in results.items():
        rfm, rfs = v['rf_stat_mean_std']; xm, xs = v['xgb_only_mean_std']
        print(f'{v["speed"]:>6d}m{v["feature_set"]:>18s}{v["n_features"]:>7d}'
              f'{rfm:>10.2f}±{rfs:<4.1f}{xm:>12.2f}±{xs:<4.1f}')

    with open(os.path.join(outdir, 't2_feature_ablation.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nSaved to {outdir}/')


if __name__ == '__main__':
    main()


  T2 — Handcrafted-feature ablation (accuracy %, mean ± std across 5 seeds)
  Regime       Feature set  #feat         RF-Stat      XGBoost-only
------------------------------------------------------------------------------------
    25m              full     35     97.65±5.3        97.29±3.8 
    25m       no_relspeed     28     98.82±2.6        97.29±3.8 
    25m    no_speed_feats     21     93.61±6.1        97.29±3.8 
    15m              full     35     72.28±9.0        72.77±13.2
    15m       no_relspeed     28     71.23±10.5       72.77±13.2
    15m    no_speed_feats     21     72.28±9.0        72.77±13.2

Saved to outputs_ablation_features/
